# Load sites from CIF

This notebook shows how to load sites directly from a [CIF file](https://www.iucr.org/resources/cif).

First, we load the data from a VASP simulation and create a trajectory for the species of interest (Lithium).

In [1]:
from gemdat import Trajectory
from gemdat.utils import DATA, VASPRUN

# Use your own data:
# VASPRUN = 'path/to/your/vasprun.xml'

trajectory = Trajectory.from_vasprun(VASPRUN)

diff_trajectory = trajectory.filter('Li')

## Loading a CIF file

Load a cif file using the `read_cif()` function. The CIF file contains the lattice, site coordinates, and any symmetry operators.

Note that the resulting structure is in P1, meaning that any symmetry is already applied.

In [2]:
# Lets try to load a CIF file
from gemdat.io import read_cif

cif_path = DATA / 'argyrodite.cif'

sites = read_cif(cif_path)
sites[0:10]

[PeriodicSite: 48h (Li) (1.816, 1.816, 0.2382) [0.183, 0.183, 0.024],
 PeriodicSite: 48h (Li) (1.816, 6.778, 5.2) [0.183, 0.683, 0.524],
 PeriodicSite: 48h (Li) (6.778, 1.816, 5.2) [0.683, 0.183, 0.524],
 PeriodicSite: 48h (Li) (6.778, 6.778, 0.2382) [0.683, 0.683, 0.024],
 PeriodicSite: 48h (Li) (9.686, 1.816, 8.108) [0.976, 0.183, 0.817],
 PeriodicSite: 48h (Li) (9.686, 6.778, 3.146) [0.976, 0.683, 0.317],
 PeriodicSite: 48h (Li) (4.724, 1.816, 3.146) [0.476, 0.183, 0.317],
 PeriodicSite: 48h (Li) (4.724, 6.778, 8.108) [0.476, 0.683, 0.817],
 PeriodicSite: 48h (Li) (8.108, 0.2382, 8.108) [0.817, 0.024, 0.817],
 PeriodicSite: 48h (Li) (8.108, 5.2, 3.146) [0.817, 0.524, 0.317]]

In [3]:
sites.lattice

Lattice
    abc : 9.924 9.924 9.924
 angles : 90.0 90.0 90.0
 volume : 977.3728410239999
      A : 9.924 0.0 6.076697417369166e-16
      B : 1.5959009175390938e-15 9.924 6.076697417369166e-16
      C : 0.0 0.0 9.924
    pbc : True True True

The simulation was performed using a (2,1,1) supercell of argyrodite. So lets correct it so the jump sites match the trajectory data.

Note that the lattice is updated in-place.

In [4]:
sites.make_supercell((2, 1, 1))
sites.lattice

Lattice
    abc : 19.848 9.924 9.924
 angles : 90.0 90.0 90.0
 volume : 1954.7456820479997
      A : 19.848 0.0 1.2153394834738332e-15
      B : 1.5959009175390938e-15 9.924 6.076697417369166e-16
      C : 0.0 0.0 9.924
    pbc : True True True

Generate the transitions and jumps from the sites loaded from the CIF file

In [5]:
transitions = trajectory.transitions_between_sites(sites=sites, floating_specie='Li')
jumps = transitions.jumps(minimal_residence=0)

In [6]:
jumps.plot_jumps_vs_time(n_parts=5)

In [7]:
jumps.plot_jumps_3d()